In [ ]:
!pip install torch torchvision transformers pandas Pillow scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from transformers import AutoTokenizer, AutoModel
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss
import matplotlib.pyplot as plt
from torch.cuda.amp import GradScaler, autocast

# 1. CONFIGURATION
BASE_PATH = r'/content/drive/MyDrive/DI725_project_dataset'
IMG_DIR = os.path.join(BASE_PATH, 'images')
MASK_DIR = os.path.join(BASE_PATH, 'masks')
LABELS_CSV = '/content/drive/MyDrive/DI725_project_dataset/binary_labels.csv'
CAPTIONS_CSV = '/content/drive/MyDrive/DI725_project_dataset/vision_captions_only.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']
SUBSET_SIZE = 500
BATCH_SIZE = 32

# --- 2. DATASET CLASS ---

class FastRemoteSensingDataset(Dataset):
    def __init__(self, labels_df, captions_df, img_dir, mask_dir, transform=None, tokenizer=None):
        self.labels_df = labels_df
        self.captions_df = captions_df
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        row = self.labels_df.iloc[idx]
        fname = row['filename']

        # Image & Mask
        image = Image.open(os.path.join(self.img_dir, fname)).convert("RGB")
        if self.transform: image = self.transform(image)

        mask = Image.open(os.path.join(self.mask_dir, fname)).convert("L")
        mask = mask.resize((14, 14))
        mask_tensor = torch.from_numpy(np.array(mask)).float() / 255.0

        # Caption
        caption = self.captions_df[self.captions_df['filename'] == fname]['vision_qwen3-vl-8b'].values[0]
        tokens = self.tokenizer(str(caption), padding='max_length', truncation=True, max_length=128, return_tensors="pt")

        labels = torch.tensor(row[CLASSES].values.astype(float), dtype=torch.float32)

        return {
            'image': image,
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'labels': labels,
            'gt_mask': mask_tensor,
            'filename': fname
        }

# --- 3. MODEL ARCHITECTURE ---

class FastGatedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision_model = models.vit_b_16(weights='DEFAULT')
        self.vision_model.heads = nn.Identity()
        self.text_model = AutoModel.from_pretrained('distilbert-base-uncased')

        self.gate = nn.Sequential(
            nn.Linear(768 + 768, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
        self.classifier = nn.Linear(768, len(CLASSES))

    def forward(self, x_img, x_ids, x_mask):
        v_feat = self.vision_model(x_img)
        t_out = self.text_model(input_ids=x_ids, attention_mask=x_mask)
        t_feat = t_out.last_hidden_state[:, 0]

        alpha = self.gate(torch.cat([v_feat, t_feat], dim=1))
        fused = alpha * v_feat + (1 - alpha) * t_feat
        logits = self.classifier(fused)

        attn_map = torch.rand(x_img.shape[0], 14, 14).to(DEVICE)
        return logits, alpha, attn_map

# --- 4. TRAINING RUN ---

def run_fast_training(epochs=2):
    # Load and Subset Data
    labels_df = pd.read_csv(LABELS_CSV)
    captions_df = pd.read_csv(CAPTIONS_CSV)

    if len(labels_df) > SUBSET_SIZE:
        labels_df = labels_df.sample(n=SUBSET_SIZE, random_state=42).reset_index(drop=True)

    train_df, val_df = train_test_split(labels_df, test_size=0.2, random_state=42)

    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(FastRemoteSensingDataset(train_df, captions_df, IMG_DIR, MASK_DIR, transform, tokenizer),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(FastRemoteSensingDataset(val_df, captions_df, IMG_DIR, MASK_DIR, transform, tokenizer),
                            batch_size=BATCH_SIZE)

    model = FastGatedModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    scaler = GradScaler() # For Mixed Precision

    print(f"Fast PoC Started. Using {SUBSET_SIZE} samples on {DEVICE}...")

    for epoch in range(epochs):
        model.train()
        for i, batch in enumerate(train_loader):
            imgs, ids, att_mask, targets, gt_masks = batch['image'].to(DEVICE, non_blocking=True), \
                                                     batch['input_ids'].to(DEVICE, non_blocking=True), \
                                                     batch['attention_mask'].to(DEVICE, non_blocking=True), \
                                                     batch['labels'].to(DEVICE, non_blocking=True), \
                                                     batch['gt_mask'].to(DEVICE, non_blocking=True)

            optimizer.zero_grad()

            with autocast(): # Mixed Precision Training
                logits, alpha, attn_maps = model(imgs, ids, att_mask)
                cls_loss = F.binary_cross_entropy_with_logits(logits, targets)
                gr_loss = F.mse_loss(attn_maps, gt_masks)
                loss = cls_loss + (0.1 * gr_loss)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if i % 5 == 0:
                print(f"E{epoch+1} B{i} | Loss: {loss.item():.4f} | Alpha: {alpha.mean().item():.2f}")

        # Metrics at Epoch End
        model.eval()
        preds_list, targets_list = [], []
        with torch.no_grad():
            for b in val_loader:
                l, _, _ = model(b['image'].to(DEVICE), b['input_ids'].to(DEVICE), b['attention_mask'].to(DEVICE))
                preds_list.append((torch.sigmoid(l) > 0.5).int().cpu().numpy())
                targets_list.append(b['labels'].numpy())

        f1 = f1_score(np.vstack(targets_list), np.vstack(preds_list), average='micro')
        print(f"\n>> Epoch {epoch+1} Validasyon Micro-F1: {f1:.4f}\n")

if __name__ == "__main__":
    run_fast_training()
    run_training_poc(epochs=3)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fast PoC Started. Using 500 samples on cuda...


/tmp/ipykernel_8152/2544044565.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # For Mixed Precision
/tmp/ipykernel_8152/2544044565.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Mixed Precision Training


E1 B0 | Loss: 0.6997 | Alpha: 0.52
E1 B5 | Loss: 0.4793 | Alpha: 0.73
E1 B10 | Loss: 0.3069 | Alpha: 0.76

>> Epoch 1 Validasyon Micro-F1: 0.7775



/tmp/ipykernel_8152/2544044565.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(): # Mixed Precision Training


E2 B0 | Loss: 0.2756 | Alpha: 0.84
E2 B5 | Loss: 0.2637 | Alpha: 0.92
E2 B10 | Loss: 0.2056 | Alpha: 0.96

>> Epoch 2 Validasyon Micro-F1: 0.8592



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Training for 3 Epochs...

Epoch 1 | Batch 0 | Loss: 0.6665 | Alpha: 0.52
Epoch 1 | Batch 20 | Loss: 0.4342 | Alpha: 0.76
Epoch 1 | Batch 40 | Loss: 0.1993 | Alpha: 0.91
Epoch 1 | Batch 60 | Loss: 0.2849 | Alpha: 0.95
Epoch 1 | Batch 80 | Loss: 0.1673 | Alpha: 0.97
Epoch 1 | Batch 100 | Loss: 0.1609 | Alpha: 0.98
Epoch 1 | Batch 120 | Loss: 0.1635 | Alpha: 0.98
Epoch 1 | Batch 140 | Loss: 0.2680 | Alpha: 0.99
Epoch 1 | Batch 160 | Loss: 0.1587 | Alpha: 0.99
Epoch 1 | Batch 180 | Loss: 0.1488 | Alpha: 0.99
Epoch 1 | Batch 200 | Loss: 0.1132 | Alpha: 0.99
Epoch 1 | Batch 220 | Loss: 0.2096 | Alpha: 0.99
Epoch 1 | Batch 240 | Loss: 0.1597 | Alpha: 1.00
Epoch 1 | Batch 260 | Loss: 0.1176 | Alpha: 0.99
Epoch 1 | Batch 280 | Loss: 0.1859 | Alpha: 1.00
Epoch 1 | Batch 300 | Loss: 0.1880 | Alpha: 1.00
Epoch 1 | Batch 320 | Loss: 0.2468 | Alpha: 1.00
Epoch 1 | Batch 340 | Loss: 0.1396 | Alpha: 1.00
Epoch 1 | Batch 360 | Loss: 0.2001 | Alpha: 1.00
Epoch 1 | Batch 380 | Loss: 0.1789 | Alp